# Day 38: Word Embeddings & Similarity

### Q1: Word2Vec on ShopSense Reviews

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the reviews dataset
try:
    reviews_df = pd.read_csv('../Day-37/data/shopsense_reviews.csv')
    print("Reviews loaded successfully.")
except Exception as e:
    print("Error loading reviews:", e)

# Preprocessing: simple tokenization
import re

def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

sentences = reviews_df['review_text'].dropna().apply(tokenize).tolist()


### (a) Train Word2Vec and Check Polysemy

In [ ]:
from gensim.models import Word2Vec
from scipy.spatial.distance import cosine

# Train Word2Vec
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Get vectors
try:
    vec_cheap = model.wv['cheap']
    vec_affordable = model.wv['affordable']
    vec_flimsy = model.wv['flimsy']
    
    print(f"Vector for 'cheap' shape: {vec_cheap.shape}") # Shows it's ONE vector
    
    # Compute cosine similarities (1 - distance)
    sim_affordable = 1 - cosine(vec_cheap, vec_affordable)
    sim_flimsy = 1 - cosine(vec_cheap, vec_flimsy)
    
    print(f"Cosine similarity ('cheap', 'affordable'): {sim_affordable:.4f}")
    print(f"Cosine similarity ('cheap', 'flimsy'): {sim_flimsy:.4f}")
except KeyError as e:
    print(f"Word missing in vocabulary: {e}")


### (b) Disambiguation System

In [ ]:
def disambiguate_cheap(sentence, model):
    tokens = tokenize(sentence)
    if 'cheap' not in tokens:
        return "Not found"
        
    # Get context words
    context_words = [w for w in tokens if w != 'cheap' and w in model.wv.key_to_index]
    
    if not context_words:
        return "Unknown vs Unknown (no context)"
        
    # Context embedding by averaging
    context_vector = np.mean([model.wv[w] for w in context_words], axis=0)
    
    # Anchors
    anchor_affordable = model.wv['affordable'] if 'affordable' in model.wv else np.zeros_like(context_vector)
    anchor_flimsy = model.wv['flimsy'] if 'flimsy' in model.wv else np.zeros_like(context_vector)
    
    # Compare
    sim_affordable = 1 - cosine(context_vector, anchor_affordable)
    sim_flimsy = 1 - cosine(context_vector, anchor_flimsy)
    
    return "affordable" if sim_affordable > sim_flimsy else "low-quality"

# Test
print("Sentence 1:", disambiguate_cheap("The phone is cheap and affordable.", model))
print("Sentence 2:", disambiguate_cheap("This plastic case feels very cheap and flimsy, it broke quickly.", model))


### (c) Window Size Comparison

In [ ]:

model_win2 = Word2Vec(sentences, vector_size=100, window=2, min_count=1, workers=4)
model_win10 = Word2Vec(sentences, vector_size=100, window=10, min_count=1, workers=4)

try:
    print("Similarity ('battery', 'camera') - Window 2 (Syntactic/local):", model_win2.wv.similarity('battery', 'camera'))
    print("Similarity ('battery', 'camera') - Window 10 (Semantic/topical):", model_win10.wv.similarity('battery', 'camera'))
except Exception as e:
    print(e)
    
# Explanation: Window 2 captures mostly localized syntactic relationships, 
# while window 10 captures broader topical/semantic relationships.


### Q2: Similarity Comparison

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sentence-transformers'])
    from sentence_transformers import SentenceTransformer

rev_a = 'incredible camera but terrible battery life'
rev_b = 'Battery drains fast, although photos are stunning'

docs = [rev_a, rev_b]


### (a) BOW and TF-IDF

In [ ]:
# BOW
vectorizer_bow = CountVectorizer(stop_words='english')
X_bow = vectorizer_bow.fit_transform(docs)
bow_sim = cosine_similarity(X_bow)[0,1]
print("BOW Similarity:", bow_sim)

# TF-IDF
vectorizer_tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = vectorizer_tfidf.fit_transform(docs)
tfidf_sim = cosine_similarity(X_tfidf)[0,1]
print("TF-IDF Similarity:", tfidf_sim)

# (b) Walk through exact word overlap for BOW failure:
# Words in A: incredible, camera, terrible, battery, life
# Words in B: battery, drains, fast, photos, stunning
# Exact overlap is ONLY 'battery'. BOW similarity will be very low despite matching sentiment.


### Word2Vec Averaging

In [ ]:
def get_sentence_vector(sentence, model):
    words = tokenize(sentence)
    words = [w for w in words if w in model.wv.key_to_index]
    if not words:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[w] for w in words], axis=0)

vec_a = get_sentence_vector(rev_a, model)
vec_b = get_sentence_vector(rev_b, model)
w2v_sim = 1 - cosine(vec_a, vec_b)
print("Word2Vec Similarity:", w2v_sim)


### Sentence-BERT

In [ ]:
# Sentence-BERT
try:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = sbert_model.encode(docs)
    sbert_sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0,0]
    print("Sentence-BERT Similarity:", sbert_sim)
except Exception as e:
    print("Could not run Sentence-BERT:", e)
    
# (c) Semantic Gap Explanation:
# - BOW/TF-IDF relies on exact matching (syntactic).
# - Word2Vec averages words that maps conceptually similar words closer.
# - Sentence-BERT creates context-aware embeddings grasping the overall meaning similarity even with varying vocabularies.
